| 算法类型 | 更新目标 (Target) | 采样方式 | 特点 |
| :--- | :--- | :--- | :--- |
| **REINFORCE** | $G_t$ (完整回报) | **Monte-Carlo** | **无偏 (Unbiased)**，但在长序列中**方差 (Variance) 极大**。 |
| **Actor-Critic** | $r_t + \gamma V(s_{t+1})$ | **TD (时序差分)** | 引入了**偏差 (Bias)**，但显著降低了**方差**。利用 Critic 网络进行自举 (Bootstrapping)。 |
| **n-step PG** | $\sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n})$ | **混合 (MC + TD)** | 介于两者之间，向前看 $n$ 步后截断，使用价值函数估计剩余部分。 |

- td: 利用下一步预测 $V(s_{t+1})$ 来更新当前预测 $V(s_t)$
    - TD 的目标（Target）：$y_t = r_t + \gamma \hat{V}(s_{t+1})$    
- mc: 必须真正地 rollout 结束，才返回更新最初的预测（开局的预测）
    - MC 的目标（Target）：$G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \dots + \gamma^{T-t} r_T$

### bias variance tradeoff

| 特性 | MC (Monte-Carlo) | TD (Temporal Difference) |
| :--- | :--- | :--- |
| **更新目标** | 真实回报 $G_t$ | 估计值 $r + \gamma V(s')$ |
| **随机性来源** | 整个序列 $(S_t \to S_T)$ | 仅当前这一步 $(S_t \to S_{t+1})$ |
| **Variance (方差)** | **高** (噪声累积) | **低** (噪声截断) |
| **Bias (偏差)** | **无** (无偏估计) | **有** (依赖不准的 Critic) |
| **收敛速度** | 慢 (因为方差大，需要更多样本平均) | 快 (方差小，且每步都能学) |

- MC 是无偏的：$G_t$ 是真实发生的累积回报。虽然每次采样的 $G_t$ 可能都不一样，但从数学期望上讲，$\mathbb{E}[G_t]$ 严格等于 真实的价值函数 $V_\pi(s_t)$。我们使用的是真实的“事实”来更新我们的估计，所以没有系统性偏差。
- TD 是有偏的：TD 的目标中包含了 $\hat{V}(s_{t+1})$。注意这个 $\hat{V}$ 是当前的价值网络（或表格）给出的估计值，而不是真实值。
    - 在训练初期，$\hat{V}$ 可能是完全随机初始化的，是错误的。
    - 当我们用一个错误的猜测（$\hat{V}(s_{t+1})$）加上当前的奖励（$r_t$）作为目标去更新 $V(s_t)$ 时，这种误差就会被传播回来。
    - 这就是 Bootstrapping（自举）：用自己的估计去更新自己的估计。这种系统性的估计误差就是 Bias。
- 即TD 用了一个“估计值”来更新另一个“估计值”，这是 bias 的来源；随着训练进行，$\hat{V}$ 会越来越准，Bias 会逐渐减小，但在训练过程中，它始终存在。

- 为什么 TD 能降低 Variance（方差）？因为 TD 截断了随机性，只看一步；而 MC 必须承受整个未来的随机性叠加。
- 方差来源于环境的随机性（Stochasticity）。主要有两个来源：
    - 状态转移的随机性：$P(s'|s,a)$
    - 动作选择的随机性：$\pi(a|s)$